# Example analysis

The pattern every notebook in this repository follows. Copy it, replace the content, keep the shape.

1. **No `def`, no `class`.** All logic lives in `src/` and is imported here. A function defined in a notebook cannot be imported or tested, so it gets copied into the next notebook and the copies diverge.
2. **Figures are saved by this notebook**, through `FigureSaver`, into `output_JEVPFN/exploration/figures/example_analysis/` as **PDF only** — and displayed inline here, which is the only raster copy there is. The setup cell clears that folder first, so no stale figure survives beside a fresh one.
3. **Never pick a colour or a size here.** `src/visualize/style.py` owns both: it fixes the A4 widths and the print settings, and this project adds its own look there so every notebook shares one.
4. **The last code cell prints a text summary**, section by section, in the same order as the sections above. `output_JEVPFN/allresults.md` is built verbatim from what the notebooks print.

In [ ]:
# Setup. Run this first, always.
import sys
from pathlib import Path

# The repository root on sys.path. WHY: Jupyter's working directory is notebooks/, so `from src...`
# does not resolve without this. The notebook runner sets cwd to the root instead, which is why the
# line is harmless there — and why it must be here, so an interactive run behaves identically.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np

from src.visualize import figures, style

# Installs the A4 sizing and the print settings, plus whatever look this project has put in
# style.py. Never set a colour or a size in a notebook — put it there instead, once.
style.apply()

# Constructing the saver CLEARS this notebook's own figure folder — before anything is drawn, and
# never another notebook's.
NOTEBOOK = "example_analysis"
from src.utils.notebook_report import begin_report, report_section, finish_report
begin_report()
report_section('Setup')
save = figures.FigureSaver(NOTEBOOK)

print(f"figures -> {save.folder}")

## Data

Replace this cell with a call into `src/data/`. The synthetic draw below exists only so the template runs on a fresh clone, before any real data is in `data/raw/`.

In [ ]:
report_section('Data')
# PLACEHOLDER. Real notebooks call src.data.loaders.load(...) instead.
rng = np.random.default_rng(20260811)
N = 2000

measurements = {
    "variant A": rng.normal(0.72, 0.05, N),
    "variant B": rng.normal(0.76, 0.04, N),
    "variant C": rng.normal(0.69, 0.07, N),
}
baseline_score = 0.70

print(f"{len(measurements)} variants, {N} draws each")

In [ ]:
report_section('Data')
# Figure 1 — distributions, drawn at the FULL A4 text width.
# Colours come from matplotlib's default cycle here, because this project has not defined a palette
# yet. Put one in src/visualize/style.py and use it from every notebook.
fig, ax = plt.subplots(figsize=style.figsize(style.WIDTH_FULL, 0.42))

for label, values in measurements.items():
    ax.hist(values, bins=40, histtype="step", linewidth=1.5, label=label)

ax.axvline(baseline_score, linestyle="--", linewidth=0.9, color="0.35")
ax.annotate("baseline", (baseline_score, 0.02), xycoords=("data", "axes fraction"),
            ha="left", va="bottom", fontsize=7, color="0.35")
ax.set_xlabel("score")
ax.set_ylabel("count")
ax.set_title("Score distribution per variant")
ax.legend(frameon=False)

save(fig, "score_distributions", caption=(
    "Step histograms of the score for three variants, 40 bins each, 2,000 draws per variant. "
    "The dashed vertical line marks the baseline score of 0.70."
))

In [ ]:
report_section('Data')
# Figure 2 — means with a spread, ordered, at HALF the A4 text width.
labels = sorted(measurements, key=lambda k: measurements[k].mean(), reverse=True)
means = [measurements[k].mean() for k in labels]
spread = [measurements[k].std() for k in labels]

fig, ax = plt.subplots(figsize=style.figsize(style.WIDTH_HALF, 0.75))
ax.bar(labels, means, yerr=spread, width=0.62)
ax.axhline(baseline_score, linestyle="--", linewidth=0.9, color="0.35")
ax.set_ylabel("mean score")
ax.set_ylim(0.5, 0.9)
ax.set_title("Mean score, ordered")

# Direct labels, so a reader never has to identify a bar by its colour alone. Offset above the
# ERROR BAR, not the bar top — otherwise the label sits inside the whisker and reads as part of it.
for x, (value, sd) in enumerate(zip(means, spread)):
    ax.annotate(f"{value:.3f}", (x, value + sd), textcoords="offset points", xytext=(0, 5),
                ha="center", fontsize=7)

save(fig, "mean_score_ranking", caption=(
    "Mean score per variant, ordered by mean, with error bars giving one standard deviation "
    "across 2,000 draws. The dashed horizontal line marks the baseline score of 0.70. Values "
    "are printed above each bar."
))

## Summary

The last code cell. Everything it prints goes into `output_JEVPFN/allresults.md` verbatim, so print the
numbers a reader would otherwise have to squint at a figure to recover — and follow the same
section order as the notebook above, so the two read the same way round.

In [ ]:
print("=" * 70)
print(f"{NOTEBOOK}: score comparison across {len(measurements)} variants")
print("=" * 70)
print(f"{N} draws per variant, baseline = {baseline_score:.3f}")
print()
print(f"{'variant':<12} {'mean':>8} {'std':>8} {'vs baseline':>12}")
for label in labels:
    values = measurements[label]
    print(f"{label:<12} {values.mean():>8.4f} {values.std():>8.4f} "
          f"{values.mean() - baseline_score:>+12.4f}")
print()
print(f"best: {labels[0]} at {measurements[labels[0]].mean():.4f}")
print()
print(save.summary())
finish_report(NOTEBOOK, save.summary())
